In [ ]:
from abc import ABC, abstractmethod


class Converter(ABC):
    @abstractmethod
    def convert(self, raw_file):
        pass

    @abstractmethod
    def get_markdown(self):
        pass

    @abstractmethod
    def get_attached_files(self):
        pass

    @abstractmethod
    def start(self):
        pass


In [ ]:
import io
from io import BytesIO, StringIO
from abc import ABC, abstractmethod

from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE, PP_PLACEHOLDER

class PptxConverter(Converter):
    def __init__(self):
        self.file_bytes = None
        self.output = []
        self.slide_count = 0
        self.image_count = 0
        self.current_slide_title = ""
        self.title_placeholder_types = {
            PP_PLACEHOLDER.TITLE,
            PP_PLACEHOLDER.CENTER_TITLE,
            PP_PLACEHOLDER.SUBTITLE
        }
        self.VERTICAL_THRESHOLD = 28.35  # 1cm = 28.35pt

    def convert(self, pptx_bytes: BytesIO) -> str:
        pptx_bytes.seek(0)
        presentation = Presentation(pptx_bytes)

        for slide in presentation.slides:
            self.slide_count += 1
            self._process_slide(slide)

        return "\n".join(self.output)

    def start(self):
        if not self.file_bytes:
            print("[WARN] No PPTX file provided")
            return

        self.file_bytes.seek(0)
        markdown = self.convert(self.file_bytes)
        self.markdown_io = StringIO(markdown)

    def get_markdown(self):
        return self.markdown_io.getvalue()

    def get_attached_files(self):
        return {}


    def _process_slide(self, slide):
        self.current_slide_title = self._get_slide_title(slide)
        self.output.append(f"\n# Слайд {self.slide_count}: {self.current_slide_title}\n")

        elements = []
        self._collect_elements(slide.shapes, elements)

        lines = []
        sorted_elements = sorted(elements, key=lambda x: (x['top'], x['left']))

        for element in sorted_elements:
            added = False
            for line in lines:
                if abs(element['top'] - line[0]['top']) <= self.VERTICAL_THRESHOLD:
                    line.append(element)
                    added = True
                    break
            if not added:
                lines.append([element])

        lines.sort(key=lambda line: line[0]['top'])
        for line in lines:
            line.sort(key=lambda x: x['left'])
            for elem in line:
                if elem['type'] == 'text':
                    self._process_text(elem['object'])
                elif elem['type'] == 'table':
                    self._process_table(elem['object'])
                elif elem['type'] == 'image':
                    self._process_image(elem['object'])
                elif elem['type'] == 'equation':
                    self._process_equation(elem['object'])

    def _collect_elements(self, shapes, elements):
        for shape in shapes:
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                self._collect_elements(shape.shapes, elements)
            else:
                if (not hasattr(shape, 'top') or not hasattr(shape, 'left') or
                    shape.top is None or shape.left is None):
                    print("[WARN] shape missing top/left")
                    continue

                if (not hasattr(shape.top, 'pt') or not hasattr(shape.left, 'pt')):
                    print("[WARN] shape missing pt")
                    continue

                try:
                    top = shape.top.pt
                    left = shape.left.pt
                except Exception as e:
                    print(f"[WARN] pt error: {e}")
                    continue

                if shape.has_text_frame:
                    text = self._get_shape_text(shape)
                    if self._is_equation(text):
                        elements.append({'type': 'equation', 'object': shape, 'top': top, 'left': left})
                    elif not self._is_duplicate_title(shape):
                        elements.append({'type': 'text', 'object': shape, 'top': top, 'left': left})

                elif shape.has_table:
                    elements.append({'type': 'table', 'object': shape.table, 'top': top, 'left': left})

                elif shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                    elements.append({'type': 'image', 'object': shape, 'top': top, 'left': left})

    def _is_duplicate_title(self, shape):
        if not shape.has_text_frame:
            return False
        text = "".join(run.text for p in shape.text_frame.paragraphs for run in p.runs)
        return text.strip() == self.current_slide_title

    def _get_slide_title(self, slide):
        for shape in slide.shapes:
            try:
                if (shape.is_placeholder and
                    shape.placeholder_format.type in self.title_placeholder_types and
                    shape.text.strip()):
                    return shape.text.strip()
            except Exception:
                pass

        top_texts = []
        for shape in slide.shapes:
            if shape.has_text_frame and shape.text.strip() and hasattr(shape.top, 'pt'):
                top_texts.append((shape.top.pt, shape.text.strip()))

        if top_texts:
            top_texts.sort(key=lambda x: x[0])
            return top_texts[0][1]

        return f"Слайд {self.slide_count}"

    def _process_text(self, shape):
        for p in shape.text_frame.paragraphs:
            text = "".join(run.text for run in p.runs).strip()
            if not text:
                continue
            level = getattr(p, 'level', 0)
            if level > 0:
                self.output.append("  " * level + "- " + text)
            else:
                self.output.append(text)

    def _process_table(self, table):
        for row in table.rows:
            cells = [self._get_cell_text(c) for c in row.cells]
            self.output.append("| " + " | ".join(cells) + " |")

        self.output.append("")

    def _get_cell_text(self, cell):
        return " ".join(
            run.text.strip()
            for p in cell.text_frame.paragraphs
            for run in p.runs
            if run.text.strip()
        )

    def _process_image(self, shape):
        self.image_count += 1
        self.output.append(f"![image {self.slide_count}.{self.image_count}]")

    def _get_shape_text(self, shape):
        return "".join(run.text for p in shape.text_frame.paragraphs for run in p.runs)

    def _is_equation(self, text):
        symbols = {'=', '∑', '∫', '√', '∞', '≈', '≠', '≤', '≥', '+', '-', '×', '÷'}
        return any(s in text for s in symbols)

    def _process_equation(self, shape):
        eq = self._get_shape_text(shape).strip()
        if eq:
            self.output.append(f"$$ {eq} $$")

In [ ]:
if __name__ == "__main__":
    with open("/content/рюкзак_остудина.pptx", "rb") as f:
        pptx_bytes = BytesIO(f.read())

    conv = PptxConverter()
    md = conv.convert(pptx_bytes)
    print(md)


# Слайд 1: умный рюкзак для осанки

$$ Дизайн-мышление $$
![image 1.1]
Остудина ксения

# Слайд 2: Содержание

$$ ВведениеПроцесс дизайн-мышления Эмпатия Фокусировка Генерация идей Выбор идейПрототипированиеТестирование Стратегия выхода на рынокЗаключение $$

# Слайд 3: введение

Проблема:
Большинство пользователей не контролируют реальный вес своего рюкзака и не замечают, что носят его неправильно
$$ ИсследованияИз 97 % детей использующих рюкзаки у 37 % из них появлялась боль в спинеВ исследовании среди подростков (12–18 лет) было обнаружено, что 74,4 % рюкзак-носителей испытывали боли в спинеДо 60% офисных работников регулярно носят ноутбуки в рюкзаках, неправильно распределяя нагрузку $$
Решение:
Создать умный рюкзак, который предотвращает проблемы со спиной:  следит, предупреждает и корректирует, чтобы проблема не успела возникнуть. Простой в использовании – понятный ребенку.

# Слайд 4: Процесс дизайнерского мышления


# Слайд 5: эмпатия

👁 Что он видит
$$ Одноклассников, сутулых

In [12]:
import time
from io import BytesIO

def measure_speed(converter, pptx_path, runs=5):
    times = []
    slides = []

    with open(pptx_path, "rb") as f:
        raw = f.read()

    for _ in range(runs):
        converter.output = []
        converter.slide_count = 0
        converter.image_count = 0

        start = time.time()
        converter.convert(BytesIO(raw))
        end = time.time()

        times.append(end - start)
        slides.append(converter.slide_count)

    avg_time = sum(times) / runs
    avg_slides = sum(slides) / runs

    return {
        "avg_time_sec": avg_time,
        "slides": avg_slides,
        "time_per_slide_sec": avg_time / avg_slides
    }

In [13]:
from pptx import Presentation

def measure_coverage(converter, pptx_path):
    with open(pptx_path, "rb") as f:
        raw = f.read()

    prs = Presentation(BytesIO(raw))

    total_text = 0
    total_tables = 0
    total_images = 0

    for slide in prs.slides:
        for shape in slide.shapes:
            if shape.has_text_frame and shape.text.strip():
                total_text += 1
            elif shape.has_table:
                total_tables += 1
            elif shape.shape_type == 13:
                total_images += 1

    converter.output = []
    converter.slide_count = 0
    converter.image_count = 0
    md = converter.convert(BytesIO(raw))

    extracted_text = sum(1 for line in md.split("\n") if line.strip())
    extracted_tables = md.count("|")
    extracted_images = md.count("Рисунок")

    return {
        "text_coverage": extracted_text / max(total_text, 1),
        "table_coverage": extracted_tables / max(total_tables, 1),
        "image_coverage": extracted_images / max(total_images, 1),
    }

In [14]:
def measure_structure(markdown_text):
    lines = markdown_text.split("\n")

    bullets = sum(1 for l in lines if l.strip().startswith("-"))
    headings = sum(1 for l in lines if l.strip().startswith("#"))
    tables = sum(1 for l in lines if l.strip().startswith("|"))
    equations = markdown_text.count("$$")

    return {
        "headings": headings,
        "bullets": bullets,
        "tables": tables,
        "equations": equations,
        "total_lines": len(lines)
    }


In [15]:
def evaluate_baseline(converter, pptx_path):
    speed = measure_speed(converter, pptx_path)
    coverage = measure_coverage(converter, pptx_path)

    with open(pptx_path, "rb") as f:
        raw = f.read()

    md = converter.convert(BytesIO(raw))
    structure = measure_structure(md)

    return {
        "speed": speed,
        "coverage": coverage,
        "structure": structure
    }


In [ ]:
conv = PptxConverter()
results = evaluate_baseline(conv, "/content/рюкзак_остудина.pptx")

for k, v in results.items():
    print(k, ":", v)

speed : {'avg_time_sec': 0.0888608455657959, 'slides': 20.0, 'time_per_slide_sec': 0.004443042278289795}
coverage : {'text_coverage': 2.4788732394366195, 'table_coverage': 0.0, 'image_coverage': 0.0}
structure : {'headings': 40, 'bullets': 8, 'tables': 0, 'equations': 44, 'total_lines': 432}


In [2]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.1 MB/s eta 0:00:00


In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.2 MB/s eta 0:00:00


## Docling-style preprocessing

In [4]:
from pptx import Presentation
from io import BytesIO
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import numpy as np

class BaselineParser:
    def __init__(self):
        self.elements = []

    def parse_slide(self, slide):
        for shape in slide.shapes:
            if not hasattr(shape, 'top') or not hasattr(shape, 'left'):
                continue

            elem = {
                "type": None,
                "top": shape.top.pt,
                "left": shape.left.pt,
                "text": "",
            }

            if shape.has_text_frame:
                elem["type"] = "text"
                elem["text"] = shape.text_frame.text.strip()

            elif shape.has_table:
                elem["type"] = "table"
                elem["text"] = "\n".join(
                    [" | ".join(c.text.strip() for c in r.cells)
                     for r in shape.table.rows]
                )

            elif shape.shape_type == 13:  # image
                elem["type"] = "image"
                elem["text"] = ""

            if elem["type"] is not None:
                self.elements.append(elem)

    def parse(self, pptx_bytes):
        self.elements = []
        pres = Presentation(BytesIO(pptx_bytes))
        for slide in pres.slides:
            self.parse_slide(slide)
        return self.elements

In [5]:
def build_graph(nodes, threshold=50):
    """
    nodes: list of extracted PPTX elements
    """
    num_nodes = len(nodes)

    # Node features: top, left + type one-hot
    type_map = {"text": 0, "table": 1, "image": 2, "equation": 3}
    x = []

    for n in nodes:
        type_feat = [0, 0, 0, 0]
        if n["type"] in type_map:
            type_feat[type_map[n["type"]]] = 1
        x.append([n["top"], n["left"]] + type_feat)

    x = torch.tensor(x, dtype=torch.float)

    # Edges: spatial proximity
    edge_index = []
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if (
                abs(nodes[i]["top"] - nodes[j]["top"]) < threshold
                or abs(nodes[i]["left"] - nodes[j]["left"]) < threshold
            ):
                edge_index.append([i, j])
                edge_index.append([j, i])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

In [6]:
class OrderGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1)  # 🔑 one score per block

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)
        return x.squeeze()  # (num_nodes,)

In [8]:
def order_elements(elements, scores):
    scores = scores.detach().cpu().numpy()
    order = np.argsort(scores)
    return [elements[i] for i in order]

def generate_markdown(elements):
    md = []

    for elem in elements:
        if elem["type"] == "text" and elem["text"]:
            md.append(elem["text"])

        elif elem["type"] == "table":
            md.append(elem["text"])

        elif elem["type"] == "image":
            md.append("![image]")

    return "\n\n".join(md)

In [9]:
if __name__ == "__main__":
    with open("/content/рюкзак_остудина.pptx", "rb") as f:
        pptx_bytes = f.read()

    # Step 1: baseline parsing
    parser = BaselineParser()
    elements = parser.parse(pptx_bytes)
    print(f"Extracted {len(elements)} elements")

    # Step 2: graph construction
    graph = build_graph(elements)
    print(f"Graph: {graph.num_nodes} nodes, {graph.num_edges} edges")

    # Step 3: neural reading order
    model = OrderGCN(in_channels=6, hidden_channels=16)
    scores = model(graph)

    # Step 4: reorder elements
    ordered_elements = order_elements(elements, scores)

    # Step 5: Markdown
    markdown = generate_markdown(ordered_elements)
    print("\n===== MARKDOWN OUTPUT =====\n")
    print(markdown)

Extracted 84 elements
Graph: 84 nodes, 3090 edges

===== MARKDOWN OUTPUT =====

![image]

эмпатия

Думает и чувствует

![image]

введение

👁 Что он видит

Содержание

Проблема: 
Большинство пользователей не контролируют реальный вес своего рюкзака и не замечают, что носят его неправильно

Первый запуск
Крупные города РФ (Москва, Санкт-Петербург)
Высокая концентрация школьников и работающих специалистов
Поведенческие признаки
Забота о здоровье и профилактике
Готовность использовать технологичные гаджеты
Желание получать мгновенную и понятную обратную связь

🎯 Потребности

Одноклассников, сутулых из-за тяжёлых рюкзаков
Неправильно надетые рюкзаки
Предупреждения врачей о вреде большой нагрузки
Собственную кривую осанку на фото и в зеркале

Карта эмпатии

Жалуется на усталость и боль в спине
Носит много вещей каждый день
Затягивает лямки “на глаз”
Иногда носит на одной лямке
Хочет следить за здоровьем, но не хочет заморачиваться

эмпатия

Прототипирование

ФОКУСИРОВКА

Прототипирование

Це

In [10]:
class NeuralPptxConverter:
    def __init__(self, model):
        self.model = model
        self.output = []
        self.slide_count = 0
        self.image_count = 0

    def convert(self, pptx_bytes: BytesIO):
        pptx_bytes.seek(0)
        raw = pptx_bytes.read()

        # --- Baseline extraction ---
        parser = BaselineParser()
        elements = parser.parse(raw)

        # slide count (для метрик скорости)
        prs = Presentation(BytesIO(raw))
        self.slide_count = len(prs.slides)

        # --- Graph ---
        graph = build_graph(elements)

        # --- GNN reading order ---
        with torch.no_grad():
            scores = self.model(graph)

        ordered_elements = order_elements(elements, scores)

        # --- Markdown ---
        markdown = generate_markdown(ordered_elements)
        self.output = markdown.split("\n")

        # image count (для совместимости)
        self.image_count = sum(1 for e in ordered_elements if e["type"] == "image")

        return markdown

In [16]:
conv_neural = NeuralPptxConverter(
    model=OrderGCN(in_channels=6, hidden_channels=16)
)

results_neural = evaluate_baseline(
    conv_neural,
    "/content/рюкзак_остудина.pptx"
)

for k, v in results_neural.items():
    print(k, ":", v)

speed : {'avg_time_sec': 0.07965812683105469, 'slides': 20.0, 'time_per_slide_sec': 0.0039829063415527345}
coverage : {'text_coverage': 3.183098591549296, 'table_coverage': 0.0, 'image_coverage': 0.0}
structure : {'headings': 0, 'bullets': 0, 'tables': 0, 'equations': 0, 'total_lines': 324}
